In [36]:
import requests

API = "https://lol.fandom.com/api.php"
HEADERS = {"User-Agent": "LOL Logo Downloader/1.0"}

def get_logo_filenames_from_category(category, limit=500):
    files = []
    cmcontinue = ""
    while True:
        params = {
            "action": "query",
            "list": "categorymembers",
            "cmtitle": f"Category:{category}",
            "cmtype": "file",
            "cmlimit": limit,
            "format": "json",
            "cmcontinue": cmcontinue
        }
        r = requests.get(API, params=params, headers=HEADERS).json()
        for f in r["query"]["categorymembers"]:
            files.append(f["title"].replace("File:", ""))
        if "continue" in r:
            cmcontinue = r["continue"]["cmcontinue"]
        else:
            break
    return files

logo_files = get_logo_filenames_from_category("Team Logo Profiles")
print(f"Found {len(logo_files)} logos")


Found 501 logos


In [51]:
import requests
import os
import re
from concurrent.futures import ThreadPoolExecutor, as_completed

API = "https://lol.fandom.com/api.php"
HEADERS = {"User-Agent": "LOL Logo Downloader/1.0"}
SAVE_DIR = r"D:\C_analyzer\LOL-game-analyzer\public\teams"
os.makedirs(SAVE_DIR, exist_ok=True)

TEAMS = [
    "Ascension Gaming","BNK FEARX","BRION","Bombers","CTBC Flying Oyster",
    "DN Freecs","DetonatioN FocusMe","Dominus Esports","ES Sharks",
    "EVOS Esports","Ever8 Winners","Excel Esports","FC Schalke 04 Esports",
    "FURIA","Fenerbahçe Esports","Gambit Esports","GiantX","Griffin",
    "INTZ","Isurus","Jin Air Green Wings","KOI","KSV eSports",
    "KaBuM! Esports","Kaos Latin Gamers","Kingzone DragonX",
    "Kongdoo Monster","Liiv SANDBOX","MEGA","MVP","Misfits Gaming",
    "Movistar KOI","Movistar R7","NRG Kia","OKSavingsBank BRION",
    "OpTic Gaming","Origen","PENTAGRAM","Papara SuperMassive",
    "Pentanet.GG","Phong Vũ Buffalo","ROX Tigers","Rogue Warriors",
    "SK Telecom T1","SeolHaeOne Prince","Seorabeol Gaming",
    "Shopify Rebellion","SinoDragon Gaming","Snake Esports","Splyce",
    "Suning","TALON","Team BattleComics","Team Dynamics",
    "ThunderTalk Gaming","Ultra Prime","Unicorns of Love.CIS","VSG",
    "Vega Squadron","Vici Gaming","Victory Five","bbq Olivers","eStar",
    "inf","paiN Gaming","İstanbul Wildcats"
 ]

ALIASES = {
    "BNK FEARX": ["Fredit BRION", "BRION"],
    "BRION": ["Fredit BRION"],
    "DN Freecs": ["Kwangdong Freecs", "Afreeca Freecs"],
    "KOI": ["Movistar KOI"],
    "KSV eSports": ["Gen.G", "Samsung Galaxy"],
    "Liiv SANDBOX": ["SANDBOX Gaming"],
    "NRG Kia": ["NRG Esports"],
    "OpTic Gaming": ["OpTic"],
    "Origen": ["OG"],
    "Splyce": ["MAD Lions"],
    "Suning": ["Weibo Gaming"],
    "SK Telecom T1": ["SKT", "T1"],
    "İstanbul Wildcats": ["Istanbul Wildcats"],
    "Phong Vũ Buffalo": ["Phong Vu Buffalo", "Buffalo"],
    "Unicorns of Love.CIS": ["Unicorns of Love"],
    "VSG": ["VSG eSports", "Victory Five"],
    "Vega Squadron": ["Vega"],
    "bbq Olivers": ["bbq"],
    "eStar": ["Estar"],
}

def norm(s):
    return re.sub(r"[^a-z0-9]", "", s.lower())

def tokens(s):
    parts = re.split(r"[^a-z0-9]+", s.lower())
    stop = {"esports", "gaming", "team", "club", "academy"}
    return [p for p in parts if p and p not in stop]

def acronym(parts):
    return "".join(p[0] for p in parts if p)

TEAM_KEYS = {norm(t): t for t in TEAMS}
FOUND = set()

def get_image_url(filename):
    r = requests.get(
        API,
        params={
            "action": "query",
            "titles": f"File:{filename}",
            "prop": "imageinfo",
            "iiprop": "url|mime",
            "iiurlwidth": 512,
            "format": "json"
        },
        headers=HEADERS,
        timeout=20
    ).json()

    for p in r["query"]["pages"].values():
        if "imageinfo" in p:
            info = p["imageinfo"][0]
            return info.get("thumburl") or info.get("url")
    return None

def pick_best_filename(team, files, alias_text=None):
    team_key = norm(team)
    base_tokens = tokens(team)
    alias_tokens = tokens(alias_text) if alias_text else []
    tks = (alias_tokens or base_tokens) or [team_key]
    acr = acronym(tks)
    best = None
    best_score = -1
    best_len = None

    for fname in files:
        fn = fname.lower()
        if not (fn.endswith(".png") or fn.endswith(".svg") or fn.endswith(".webp")):
            continue
        fname_norm = norm(fname)
        if not all(tok in fname_norm for tok in tks):
            continue
        score = sum(len(tok) for tok in tks)
        if team_key in fname_norm:
            score += 10
        if acr and acr in fname_norm:
            score += 5
        if score > best_score or (score == best_score and (best_len is None or len(fname_norm) < best_len)):
            best = fname
            best_score = score
            best_len = len(fname_norm)
    return best

SEARCH_CACHE = {}
def search_file_titles(query):
    if query in SEARCH_CACHE:
        return SEARCH_CACHE[query]
    r = requests.get(
        API,
        params={
            "action": "query",
            "list": "search",
            "srnamespace": 6,
            "srlimit": 20,
            "srsearch": f"{query} logo",
            "format": "json"
        },
        headers=HEADERS,
        timeout=20
    ).json()
    titles = [s["title"].replace("File:", "") for s in r.get("query", {}).get("search", [])]
    SEARCH_CACHE[query] = titles
    return titles

# Build best match per team
TEAM_TO_FILE = {}
for team in TEAMS:
    # 1) try category list with team name
    best = pick_best_filename(team, logo_files)
    # 2) try aliases against category list
    if not best:
        for alias in ALIASES.get(team, []):
            best = pick_best_filename(team, logo_files, alias_text=alias)
            if best:
                break
    # 3) try search API with team name and aliases
    if not best:
        candidates = search_file_titles(team)
        best = pick_best_filename(team, candidates)
    if not best:
        for alias in ALIASES.get(team, []):
            candidates = search_file_titles(alias)
            best = pick_best_filename(team, candidates, alias_text=alias)
            if best:
                break
    TEAM_TO_FILE[team] = best

def process_team(team):
    fname = TEAM_TO_FILE.get(team)
    if not fname:
        return (team, "no-match")

    out_name = norm(team) + ".png"
    out_path = os.path.join(SAVE_DIR, out_name)
    if os.path.exists(out_path):
        return (team, "exists")

    url = get_image_url(fname)
    if not url:
        return (team, "no-url")

    r = requests.get(url, headers=HEADERS, timeout=25)
    if r.status_code != 200:
        return (team, f"http-{r.status_code}")

    with open(out_path, "wb") as f:
        f.write(r.content)
    return (team, "ok")

# ---- DOWNLOAD ----
results = []
with ThreadPoolExecutor(max_workers=8) as executor:
    futures = [executor.submit(process_team, t) for t in TEAMS]
    for future in as_completed(futures):
        results.append(future.result())

for team, status in results:
    if status == "ok":
        FOUND.add(team)
        print(f"[OK] {team}")
    elif status == "exists":
        FOUND.add(team)
        print(f"[SKIP] {team}")
    elif status == "no-match":
        print(f"[MISS] {team} (no filename match)")
    else:
        print(f"[FAIL] {team} ({status})")

print("\n=== FINAL RESULT ===")
print(f"PNG logos ready: {len(FOUND)} / {len(TEAMS)}")

missing = [t for t in TEAMS if t not in FOUND]
if missing:
    print("\nStill missing:")
    for m in missing:
        print("-", m)

[SKIP] DetonatioN FocusMe
[SKIP] DN Freecs
[SKIP] BRION
[OK] Dominus Esports
[SKIP] Excel Esports
[SKIP] FC Schalke 04 Esports
[SKIP] FURIA
[OK] EVOS Esports
[SKIP] Gambit Esports
[OK] CTBC Flying Oyster
[SKIP] Griffin
[OK] Ascension Gaming
[OK] ES Sharks
[OK] Bombers
[OK] INTZ
[OK] Fenerbahçe Esports
[SKIP] KaBuM! Esports
[SKIP] Kaos Latin Gamers
[SKIP] Kingzone DragonX
[OK] GiantX
[SKIP] Liiv SANDBOX
[SKIP] MEGA
[OK] Jin Air Green Wings
[SKIP] Misfits Gaming
[OK] Isurus
[OK] Kongdoo Monster
[OK] Movistar KOI
[SKIP] OKSavingsBank BRION
[SKIP] OpTic Gaming
[SKIP] Origen
[SKIP] PENTAGRAM
[SKIP] Papara SuperMassive
[SKIP] Pentanet.GG
[OK] KOI
[OK] Movistar R7
[OK] KSV eSports
[OK] Phong Vũ Buffalo
[OK] BNK FEARX
[OK] Rogue Warriors
[SKIP] Shopify Rebellion
[OK] Ever8 Winners
[OK] ROX Tigers
[OK] SeolHaeOne Prince
[OK] Seorabeol Gaming
[SKIP] TALON
[OK] MVP
[SKIP] Team Dynamics
[SKIP] ThunderTalk Gaming
[SKIP] Ultra Prime
[OK] SinoDragon Gaming
[OK] NRG Kia
[OK] Snake Esports
[SKIP] Vici 

In [56]:
existing_files = {
    os.path.splitext(f)[0]
    for f in os.listdir(SAVE_DIR)
    if f.lower().endswith(".png")
}

missing_teams = [
    team for team in TEAMS
    if norm(team) not in existing_files
]

print("Missing teams:")
for t in missing_teams:
    print("-", t)


Missing teams:


In [53]:
def retry_team(team):
    queries = [team] + ALIASES.get(team, [])

    for q in queries:
        candidates = search_file_titles(q)
        if not candidates:
            continue

        fname = pick_best_filename(team, candidates, alias_text=q)
        if not fname:
            continue

        url = get_image_url(fname)
        if not url:
            continue

        r = requests.get(url, headers=HEADERS, timeout=20)
        if r.status_code != 200:
            continue

        out_path = os.path.join(SAVE_DIR, norm(team) + ".png")
        with open(out_path, "wb") as f:
            f.write(r.content)

        return team, "ok"

    return team, "fail"


In [54]:
with ThreadPoolExecutor(max_workers=8) as executor:
    results = executor.map(retry_team, missing_teams)

for team, status in results:
    print(f"[{status.upper()}] {team}")


[OK] Ascension Gaming
[OK] KSV eSports
[OK] SK Telecom T1


In [55]:
import requests
import os
import re

API = "https://lol.fandom.com/api.php"
HEADERS = {"User-Agent": "LOL Logo Downloader/1.0"}

SAVE_DIR = r"D:\C_analyzer\LOL-game-analyzer\public\teams"
os.makedirs(SAVE_DIR, exist_ok=True)

OUT_PATH = os.path.join(SAVE_DIR, "t1.png")

def norm(s):
    return re.sub(r"[^a-z0-9]", "", s.lower())

def search_files():
    r = requests.get(
        API,
        params={
            "action": "query",
            "list": "search",
            "srnamespace": 6,
            "srlimit": 20,
            "srsearch": "T1 logo",
            "format": "json"
        },
        headers=HEADERS,
        timeout=20
    ).json()

    return [
        s["title"].replace("File:", "")
        for s in r.get("query", {}).get("search", [])
    ]

def get_image_url(filename):
    r = requests.get(
        API,
        params={
            "action": "query",
            "titles": f"File:{filename}",
            "prop": "imageinfo",
            "iiprop": "url|mime",
            "iiurlwidth": 512,
            "format": "json"
        },
        headers=HEADERS,
        timeout=20
    ).json()

    for p in r["query"]["pages"].values():
        if "imageinfo" in p:
            return p["imageinfo"][0].get("thumburl") or p["imageinfo"][0].get("url")
    return None

files = search_files()

best = None
best_score = -1

for f in files:
    fn = f.lower()

    # 🚫 reject old SKT
    if "skt" in fn or "sk telecom" in fn:
        continue

    # PNG only
    if not fn.endswith(".png"):
        continue

    score = 0
    if "t1" in fn:
        score += 10
    if "logo" in fn:
        score += 5
    if "2020" in fn or "2021" in fn or "2022" in fn:
        score += 3

    if score > best_score:
        best = f
        best_score = score

if not best:
    print("[FAIL] No modern T1 logo found")
    exit()

url = get_image_url(best)
if not url:
    print("[FAIL] Image URL not found")
    exit()

r = requests.get(url, headers=HEADERS, timeout=25)
if r.status_code != 200:
    print("[FAIL] Download error")
    exit()

with open(OUT_PATH, "wb") as f:
    f.write(r.content)

print(f"[OK] Modern T1 logo saved as t1.png")


[OK] Modern T1 logo saved as t1.png


In [57]:
import os
import re
import requests
from PIL import Image
from io import BytesIO

API = "https://lol.fandom.com/api.php"
HEADERS = {"User-Agent": "LOL Logo Downloader/1.0"}
DIR = r"D:\C_analyzer\LOL-game-analyzer\public\teams"

os.makedirs(DIR, exist_ok=True)

TEAMS = {
    "100 Thieves": "100thieves.png",
    "Astralis": "astralis.png",
    "Bilibili Gaming": "bilibili_gaming.png",
    "Cloud9": "cloud9.png",
    "DRX": "drx.png",
    "Dignitas": "dignitas.png",
    "Dplus KIA": "dplus_kia.png",
    "Evil Geniuses": "evil_geniuses_na.png",
    "FlyQuest": "flyquest.png",
    "Fnatic": "fnatic.png",
    "G2 Esports": "g2esports.png",
    "Gen.G": "gen_g.png",
    "Hanwha Life Esports": "hanwha_life_esports.png",
    "Immortals": "immortals.png",
    "Invictus Gaming": "invictus_gaming.png",
    "JD Gaming": "jd_gaming.png",
    "KT Rolster": "kt_rolster.png",
    "Karmine Corp": "karmine_corp.png",
    "LNG Esports": "lngesports.png",
    "LOUD": "loud.png",
    "MAD Lions KOI": "mad_lions.png",
    "Ninjas in Pyjamas": "ninjas_in_pyjamas_cn.png",
    "Nongshim RedForce": "nongshim_redforce.png",
    "Rare Atom": "rare_atom.png",
    "Rogue": "rogue.png",
    "SK Gaming": "sk_gaming.png",
    "TSM": "tsm.png",
    "Team BDS": "team_bds.png",
    "Team Liquid": "team_liquid.png",
    "Team Vitality": "teamvitality.png",
    "Team WE": "team_we.png",
    "Top Esports": "top_esports.png",
    "Weibo Gaming": "weibo_gaming.png"
}

def norm(s):
    return re.sub(r"[^a-z0-9]", "", s.lower())

def search_logo(team):
    r = requests.get(
        API,
        params={
            "action": "query",
            "list": "search",
            "srnamespace": 6,
            "srlimit": 15,
            "srsearch": f"{team} logo",
            "format": "json"
        },
        headers=HEADERS,
        timeout=20
    ).json()

    for s in r.get("query", {}).get("search", []):
        t = s["title"].lower()
        if "skt" in t:
            continue
        if t.endswith((".png", ".webp")):
            return s["title"].replace("File:", "")
    return None

def get_image_url(filename):
    r = requests.get(
        API,
        params={
            "action": "query",
            "titles": f"File:{filename}",
            "prop": "imageinfo",
            "iiprop": "url",
            "format": "json"
        },
        headers=HEADERS,
        timeout=20
    ).json()

    for p in r["query"]["pages"].values():
        if "imageinfo" in p:
            return p["imageinfo"][0]["url"]
    return None

existing = {norm(f): f for f in os.listdir(DIR)}

for team, target in TEAMS.items():
    target_path = os.path.join(DIR, target)

    if os.path.exists(target_path):
        print(f"[OK] {target}")
        continue

    # rename if similar file exists
    key = norm(team)
    for k, f in existing.items():
        if key in k:
            os.rename(os.path.join(DIR, f), target_path)
            print(f"[RENAME] {f} -> {target}")
            break
    else:
        # download
        file = search_logo(team)
        if not file:
            print(f"[MISS] {team}")
            continue

        url = get_image_url(file)
        if not url:
            print(f"[FAIL] {team}")
            continue

        r = requests.get(url, headers=HEADERS, timeout=25)
        img = Image.open(BytesIO(r.content)).convert("RGBA")
        img.save(target_path, "PNG")
        print(f"[DOWNLOADED] {target}")

print("\nDONE.")


[DOWNLOADED] 100thieves.png
[DOWNLOADED] astralis.png
[DOWNLOADED] bilibili_gaming.png
[DOWNLOADED] cloud9.png
[DOWNLOADED] drx.png
[DOWNLOADED] dignitas.png
[DOWNLOADED] dplus_kia.png
[DOWNLOADED] evil_geniuses_na.png
[DOWNLOADED] flyquest.png
[DOWNLOADED] fnatic.png
[DOWNLOADED] g2esports.png
[DOWNLOADED] gen_g.png
[DOWNLOADED] hanwha_life_esports.png
[DOWNLOADED] immortals.png
[DOWNLOADED] invictus_gaming.png
[DOWNLOADED] jd_gaming.png
[DOWNLOADED] kt_rolster.png
[DOWNLOADED] karmine_corp.png
[DOWNLOADED] lngesports.png
[DOWNLOADED] loud.png
[DOWNLOADED] mad_lions.png
[DOWNLOADED] ninjas_in_pyjamas_cn.png
[RENAME] nongshim_redforce.webp -> nongshim_redforce.png
[DOWNLOADED] rare_atom.png
[RENAME] roguewarriors.png -> rogue.png
[DOWNLOADED] sk_gaming.png
[DOWNLOADED] tsm.png
[DOWNLOADED] team_bds.png
[DOWNLOADED] team_liquid.png
[DOWNLOADED] teamvitality.png
[DOWNLOADED] team_we.png
[DOWNLOADED] top_esports.png
[DOWNLOADED] weibo_gaming.png

DONE.


In [59]:
import os
import requests

SAVE_DIR = r"D:\C_analyzer\LOL-game-analyzer\public\teams"
SIZE_LIMIT = 15 * 1024  # 15 KB
HEADERS = {"User-Agent": "LOL-Logo-Fetcher/2.0"}

TEAMS = {
    "Astralis (esports)": "astralis.png",
    "DRX (esports)": "drx.png",
    "Evil Geniuses": "evil_geniuses_na.png",
    "Gen.G": "gen_g.png",
    "Hanwha Life Esports": "hanwha_life_esports.png",
    "Immortals (esports)": "immortals.png",
    "JD Gaming": "jd_gaming.png",
    "Nongshim RedForce": "nongshim_redforce.png",
    "Rogue (esports)": "rogue.png",
    "SK Gaming": "sk_gaming.png",
    "Team BDS": "team_bds.png",
    "Top Esports": "top_esports.png",
}

def fetch_wiki_image(page_title):
    url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{page_title.replace(' ', '_')}"
    r = requests.get(url, headers=HEADERS, timeout=20)
    if r.status_code != 200:
        return None
    data = r.json()
    return data.get("originalimage", {}).get("source")

for page, filename in TEAMS.items():
    path = os.path.join(SAVE_DIR, filename)

    if not os.path.exists(path):
        continue

    if os.path.getsize(path) >= SIZE_LIMIT:
        continue

    os.remove(path)
    print(f"[DELETE] {filename}")

    img_url = fetch_wiki_image(page)
    if not img_url:
        print(f"[MISS] {filename}")
        continue

    img = requests.get(img_url, headers=HEADERS, timeout=25)
    if img.status_code != 200:
        print(f"[FAIL] {filename}")
        continue

    with open(path, "wb") as f:
        f.write(img.content)

    print(f"[REPLACED] {filename} ({len(img.content)//1024} KB)")

print("\nDONE. Wikipedia-quality logos restored.")


[DELETE] astralis.png
[MISS] astralis.png
[DELETE] drx.png
[REPLACED] drx.png (69 KB)
[DELETE] evil_geniuses_na.png
[REPLACED] evil_geniuses_na.png (14 KB)
[DELETE] gen_g.png
[REPLACED] gen_g.png (7 KB)
[DELETE] hanwha_life_esports.png
[REPLACED] hanwha_life_esports.png (15 KB)
[DELETE] immortals.png
[REPLACED] immortals.png (11 KB)
[DELETE] jd_gaming.png
[REPLACED] jd_gaming.png (10 KB)
[DELETE] sk_gaming.png
[REPLACED] sk_gaming.png (16 KB)
[DELETE] team_bds.png
[MISS] team_bds.png
[DELETE] top_esports.png
[REPLACED] top_esports.png (16 KB)

DONE. Wikipedia-quality logos restored.


In [74]:
from PIL import Image
import os

TEAM_DIR = r"D:\C_analyzer\LOL-game-analyzer\public\teams"
SUPPORTED = (".jpg", ".jpeg", ".webp", ".bmp")

for file in os.listdir(TEAM_DIR):
    path = os.path.join(TEAM_DIR, file)
    name, ext = os.path.splitext(file)

    if ext.lower() in SUPPORTED:
        out_path = os.path.join(TEAM_DIR, name + ".png")

        try:
            with Image.open(path) as img:
                # Convert to RGBA to preserve transparency if possible
                img = img.convert("RGBA")
                img.save(out_path, "PNG", optimize=True)

            os.remove(path)
            print(f"[CONVERTED] {file} → {name}.png")

        except Exception as e:
            print(f"[FAIL] {file}: {e}")

print("\nAll possible images converted to PNG.")


[CONVERTED] roguewarriors.webp → roguewarriors.png

All possible images converted to PNG.


In [63]:
import os

TEAM_DIR = r"D:\C_analyzer\LOL-game-analyzer\public\teams"
MIN_SIZE_KB = 5

small_files = []

for file in os.listdir(TEAM_DIR):
    if file.lower().endswith(".png"):
        path = os.path.join(TEAM_DIR, file)
        size_kb = os.path.getsize(path) / 1024

        if size_kb < MIN_SIZE_KB:
            small_files.append((file, round(size_kb, 2)))
            print(f"[LOW] {file} — {round(size_kb, 2)} KB")

print("\n=== SUMMARY ===")
print(f"Low-quality PNGs (< {MIN_SIZE_KB} KB): {len(small_files)}")

if not small_files:
    print("All PNG icons look acceptable.")


[LOW] bbqolivers.png — 1.38 KB
[LOW] dignitas.png — 1.55 KB
[LOW] dominusesports.png — 0.99 KB
[LOW] dplus_kia.png — 2.13 KB
[LOW] essharks.png — 1.95 KB
[LOW] estar.png — 1.88 KB
[LOW] evosesports.png — 0.95 KB
[LOW] excelesports.png — 2.81 KB
[LOW] gen_g.png — 2.38 KB
[LOW] giantx.png — 0.96 KB
[LOW] immortals.png — 4.92 KB
[LOW] intz.png — 1.14 KB
[LOW] isurus.png — 1.7 KB
[LOW] jinairgreenwings.png — 1.8 KB
[LOW] kongdoomonster.png — 1.36 KB
[LOW] mvp.png — 1.17 KB
[LOW] opticgaming.png — 4.83 KB
[LOW] rare_atom.png — 2.74 KB
[LOW] rogue.png — 3.5 KB
[LOW] roxtigers.png — 1.56 KB
[LOW] schalke_04.png — 3.52 KB
[LOW] seolhaeoneprince.png — 1.5 KB
[LOW] seorabeolgaming.png — 1.19 KB
[LOW] sinodragongaming.png — 1.01 KB
[LOW] sktelecomt1.png — 1.86 KB
[LOW] snakeesports.png — 1.48 KB
[LOW] teambattlecomics.png — 1.48 KB
[LOW] top_esports.png — 2.93 KB
[LOW] vegasquadron.png — 0.89 KB

=== SUMMARY ===
Low-quality PNGs (< 5 KB): 29


In [64]:
import os

TEAM_DIR = r"D:\C_analyzer\LOL-game-analyzer\public\teams"
MIN_SIZE_KB = 15

# Canonical / valid team filenames (lowercase, exact)
VALID_TEAMS = {
    "100thieves.png",
    "astralis.png",
    "bilibili_gaming.png",
    "cloud9.png",
    "drx.png",
    "dignitas.png",
    "dplus_kia.png",
    "evil_geniuses_na.png",
    "flyquest.png",
    "fnatic.png",
    "g2esports.png",
    "gen_g.png",
    "hanwha_life_esports.png",
    "immortals.png",
    "invictus_gaming.png",
    "jd_gaming.png",
    "kt_rolster.png",
    "karmine_corp.png",
    "lngesports.png",
    "loud.png",
    "mad_lions.png",
    "ninjas_in_pyjamas_cn.png",
    "nongshim_redforce.png",
    "rare_atom.png",
    "rogue.png",
    "sk_gaming.png",
    "tsm.png",
    "team_bds.png",
    "team_liquid.png",
    "teamvitality.png",
    "team_we.png",
    "top_esports.png",
    "weibo_gaming.png",
}

for file in os.listdir(TEAM_DIR):
    if not file.lower().endswith(".png"):
        continue

    path = os.path.join(TEAM_DIR, file)
    size_kb = os.path.getsize(path) / 1024

    if size_kb < MIN_SIZE_KB and file.lower() not in VALID_TEAMS:
        os.remove(path)
        print(f"[DELETE] {file} — {round(size_kb, 2)} KB (not a valid team name)")

print("\nCleanup finished.")


[DELETE] bbqolivers.png — 1.38 KB (not a valid team name)
[DELETE] clutch_gaming.png — 6.38 KB (not a valid team name)
[DELETE] dominusesports.png — 0.99 KB (not a valid team name)
[DELETE] essharks.png — 1.95 KB (not a valid team name)
[DELETE] estar.png — 1.88 KB (not a valid team name)
[DELETE] evosesports.png — 0.95 KB (not a valid team name)
[DELETE] excelesports.png — 2.81 KB (not a valid team name)
[DELETE] giantx.png — 0.96 KB (not a valid team name)
[DELETE] intz.png — 1.14 KB (not a valid team name)
[DELETE] isurus.png — 1.7 KB (not a valid team name)
[DELETE] jinairgreenwings.png — 1.8 KB (not a valid team name)
[DELETE] kongdoomonster.png — 1.36 KB (not a valid team name)
[DELETE] mvp.png — 1.17 KB (not a valid team name)
[DELETE] nrgkia.png — 10.95 KB (not a valid team name)
[DELETE] opticgaming.png — 4.83 KB (not a valid team name)
[DELETE] paingaming.png — 6.2 KB (not a valid team name)
[DELETE] pentagram.png — 8.41 KB (not a valid team name)
[DELETE] phongvbuffalo.png —